In [3]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Settings
# ============================================================
dataset_path = "../data/naffinity_dataset.csv"
CORR_METHOD = "spearman"
CORR_THRESHOLD = 0.75
RANDOM_STATE = 42

# Best hyperparameters (from your GridSearchCV selected by mean_test_f1)
RF_PARAMS = dict(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    bootstrap=False,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1

)
# ============================================================
# Fold-safe preprocessing components (fit on TRAIN ONLY)
# ============================================================
class NumericCleaner(BaseEstimator, TransformerMixin):
    """
    Transform:
      - coerce to numeric
      - inf/-inf -> NaN -> 0
      - clip to float32 range to avoid overflow in estimators
    """
    def __init__(self, clip_to_float32=True):
        self.clip_to_float32 = clip_to_float32

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X = X.apply(pd.to_numeric, errors="coerce")
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        if self.clip_to_float32:
            f32max = np.finfo(np.float32).max
            X = X.clip(lower=-f32max, upper=f32max)
        return X


class DropConstantColumns(BaseEstimator, TransformerMixin):
    """Fit on TRAIN only: drop columns with <=1 unique value in TRAIN."""
    def __init__(self):
        self.keep_cols_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        nunique = X.nunique(dropna=False)
        self.keep_cols_ = nunique[nunique > 1].index.tolist()
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.reindex(columns=self.keep_cols_, fill_value=0.0)


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Fit on TRAIN only:
      - compute correlation matrix (Spearman)
      - drop one of each pair with |corr| > threshold (upper triangle)
    """
    def __init__(self, threshold=0.75, method="spearman"):
        self.threshold = float(threshold)
        self.method = method
        self.keep_cols_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        if X.shape[1] <= 1:
            self.keep_cols_ = X.columns.tolist()
            return self

        corr = X.corr(method=self.method).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [c for c in upper.columns if (upper[c] > self.threshold).any()]
        self.keep_cols_ = [c for c in X.columns if c not in set(to_drop)]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.reindex(columns=self.keep_cols_, fill_value=0.0)


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv(dataset_path)

# Split train/test (test is Stratified)
train_df = df[df["Split"] == "Train"].copy()
test_df  = df[df["Split"] == "Stratified"].copy()

# Define features and labels
drop_cols = ["AffinityValue", "Label", "Folder", "Split"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X_train_raw = train_df[feature_cols].reset_index(drop=True)
y_train = train_df["Label"].astype(int).reset_index(drop=True)

X_test_raw = test_df[feature_cols].reset_index(drop=True)
y_test = test_df["Label"].astype(int).reset_index(drop=True)

print(f"Dataset: {dataset_path}")
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)} | Raw features: {len(feature_cols)}")

# ============================================================
# Final pipeline (fit preprocessing on TRAIN ONLY, then model)
# ============================================================
rf_pipeline = Pipeline([
    ("clean", NumericCleaner(clip_to_float32=True)),
    ("drop_const", DropConstantColumns()),
    ("corr", CorrelationFilter(threshold=CORR_THRESHOLD, method=CORR_METHOD)),
    ("clf", RandomForestClassifier(**RF_PARAMS)),
])

# ============================================================
# Train model
# ============================================================
rf_pipeline.fit(X_train_raw, y_train)

# ============================================================
# Predict using optimized threshold
# ============================================================
OPTIMAL_THRESHOLD = 0.38

rf_probs = rf_pipeline.predict_proba(X_test_raw)[:, 1]
rf_preds = (rf_probs >= OPTIMAL_THRESHOLD).astype(int)

# ============================================================
# Evaluation
# ============================================================
print(
    f"\n🎯 Evaluation Metrics "
    f"(Random Forest on Stratified Test Set, threshold={OPTIMAL_THRESHOLD})"
)

print(f"  Accuracy : {accuracy_score(y_test, rf_preds):.4f}")
print(f"  Precision: {precision_score(y_test, rf_preds, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_test, rf_preds, zero_division=0):.4f}")
print(f"  F1 Score : {f1_score(y_test, rf_preds):.4f}")

label_map = {
    0: "Weak Binders (≥1 μM)",
    1: "Strong Binders (<1 μM)"
}

y_test_named = y_test.map(label_map)
rf_named = pd.Series(rf_preds).map(label_map)

print(
    "\n"
    + classification_report(
        y_test_named,
        rf_named,
        digits=4,
        zero_division=0
    )
)

# ============================================================
# Confusion matrix
# ============================================================
cm = confusion_matrix(
    y_test,
    rf_preds,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

accuracy_weak = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

accuracy_strong = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0.0
)

print(
    "\n📊 Confusion Matrix "
    "(rows=true, cols=pred; order=[Weak, Strong])"
)
print(cm)

print("\n📊 Per-Class Accuracy:")
print(
    f"  Weak Binders (≥1 μM):   "
    f"{accuracy_weak:.4f}"
)
print(
    f"  Strong Binders (<1 μM): "
    f"{accuracy_strong:.4f}"
)

# ============================================================
# Additional probability diagnostics
# ============================================================
print("\n📈 Probability Summary")
print(
    f"Mean predicted strong-binding probability: "
    f"{rf_probs.mean():.4f}"
)

print(
    f"Predicted strong binders @ threshold "
    f"{OPTIMAL_THRESHOLD}: "
    f"{rf_preds.sum()} / {len(rf_preds)} "
    f"({100 * rf_preds.mean():.1f}%)"
)

Dataset: ../data/naffinity_dataset.csv
Train rows: 1062 | Test rows: 117 | Raw features: 302

🎯 Evaluation Metrics (Random Forest on Stratified Test Set, threshold=0.38)
  Accuracy : 0.8120
  Precision: 0.6792
  Recall   : 0.8780
  F1 Score : 0.7660

                        precision    recall  f1-score   support

Strong Binders (<1 μM)     0.6792    0.8780    0.7660        41
  Weak Binders (≥1 μM)     0.9219    0.7763    0.8429        76

              accuracy                         0.8120       117
             macro avg     0.8006    0.8272    0.8044       117
          weighted avg     0.8369    0.8120    0.8159       117


📊 Confusion Matrix (rows=true, cols=pred; order=[Weak, Strong])
[[59 17]
 [ 5 36]]

📊 Per-Class Accuracy:
  Weak Binders (≥1 μM):   0.7763
  Strong Binders (<1 μM): 0.8780

📈 Probability Summary
Mean predicted strong-binding probability: 0.3790
Predicted strong binders @ threshold 0.38: 53 / 117 (45.3%)
